# 06 — Random Forest



Primer modelo clásico de comparación, partiendo de la misma señal limpia que el notebook 03. Random Forest admite de forma nativa salidas multi-etiqueta: se entrena un único bosque sobre la matriz de 132 características, sin necesidad de envolverlo en un clasificador auxiliar. Sin balanceo de clases ni ajuste de umbral, igual que el 03 — la comparación es entre arquitecturas en igualdad de condiciones.

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos, welch
from scipy.stats import skew, kurtosis

from sklearn.ensemble import RandomForestClassifier

print(sys.executable)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

C:\Users\Usuario\anaconda3\envs\tfg\python.exe


## 2. Configuración general



`NOTEBOOK_NAME` identifica la fila en la tabla comparativa. `PREP_VERSION` aquí es `features_clasicas` — distinto del usado por las CNN, porque el caché guarda vectores de características, no tensores STFT. Los tres modelos clásicos (06, 07, 08) comparten este mismo caché.

In [2]:
NOTEBOOK_NAME = "06_random_forest"
PREP_VERSION  = "features_clasicas"

DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / PREP_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

## 3. Carga de metadatos y etiquetas multi-etiqueta

In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)

## 4. Split por paciente

In [4]:
train_df = df[df["strat_fold"].isin(range(1, 9))].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

assert set(train_df.patient_id) & set(val_df.patient_id) == set()
assert set(train_df.patient_id) & set(test_df.patient_id) == set()
assert set(val_df.patient_id)  & set(test_df.patient_id)  == set()

print("Tamaños:", len(train_df), len(val_df), len(test_df))

Tamaños: 17084 2146 2158


## 5. Preprocesado de la señal ECG (limpieza)



Idéntico al `preprocess_ecg` del notebook 03: filtro paso-banda de fase cero (0.5-40Hz), notch a 50Hz y normalización z-score por derivación. Se reutiliza tal cual — el punto de partida de los modelos clásicos es la misma señal limpia que usa la CNN, solo cambia lo que se extrae después.

In [5]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    """Descarta derivaciones corruptas: con NaN/Inf o completamente planas (fallo de electrodo)."""
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    """Limpia una derivación (1D): paso-banda + notch + normalización z-score."""
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    """Limpia las 12 derivaciones de un ECG. Descarta el registro si más de la mitad fallan."""
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned

## 6. Extracción de características clásicas



Por cada derivación se calculan 11 características: 7 estadísticos temporales (media, desviación típica, mínimo, máximo, RMS, asimetría, curtosis), la tasa de cruces por cero, y la energía relativa en 3 bandas de frecuencia (baja 0.5-4Hz, media 4-15Hz, alta 15-40Hz) obtenidas de la densidad espectral de potencia (Welch). Con 12 derivaciones son 132 características por ECG — un vector plano que sí pueden recibir Random Forest, XGBoost y Regresión Logística, frente al tensor (33,64,12) de la CNN.

In [6]:
BANDS = [(0.5, 4.0), (4.0, 15.0), (15.0, 40.0)]

def extract_features_lead(x):
    """Convierte una derivación 1D ya limpia en un vector de 11 características."""
    rms = np.sqrt(np.mean(x**2))
    zero_crossings = np.sum(np.diff(np.sign(x)) != 0) / len(x)

    freqs, psd = welch(x, fs=FS, nperseg=min(256, len(x)))
    total_power = psd.sum()
    band_powers = []
    for lo, hi in BANDS:
        mask = (freqs >= lo) & (freqs < hi)
        p = psd[mask].sum()
        band_powers.append(p / total_power if total_power > 1e-12 else 0.0)

    sk = skew(x)
    sk = sk if np.isfinite(sk) else 0.0
    ku = kurtosis(x)
    ku = ku if np.isfinite(ku) else 0.0

    return np.array([
        x.mean(), x.std(), x.min(), x.max(), rms,
        sk, ku,
        zero_crossings,
        *band_powers,
    ], dtype=np.float32)

FEATURE_NAMES = ["media", "std", "min", "max", "rms", "skew", "kurtosis", "cruces_cero",
                  "energia_baja", "energia_media", "energia_alta"]

def ecg_to_features(signal_2d):
    """signal_2d: (n_muestras, 12) -> vector (132,), 11 características x 12 derivaciones."""
    feats = [extract_features_lead(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.concatenate(feats)

## 7. Generación del dataset de características con caché en disco



Mismo patrón de caché que en los notebooks 01-05, pero aquí cada archivo `.npy` guarda un vector de 132 valores en vez de un tensor. Al compartir `PREP_VERSION`, los notebooks 06, 07 y 08 reutilizan el mismo caché entre sí — solo se calcula una vez.

In [7]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_features_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    feats = ecg_to_features(clean)
    np.save(cp, feats)
    return feats

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        f = get_features_cached(row["filename_lr"])
        if f is None:
            continue
        Xs.append(f)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_train, Y_train = build_split(train_df)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)

print("Formas:", X_train.shape, X_val.shape, X_test.shape)

Formas: (17084, 132) (2146, 132) (2158, 132)


## 8. Modelo: Random Forest



300 árboles, sin límite de profundidad (el propio `min_samples_leaf` actúa de regularización), usando todos los núcleos disponibles.

In [8]:
model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1,
)
model.fit(X_train, Y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 9. Evaluación sobre el conjunto de test

In [9]:
def get_prob_matrix(clf, X):
    """Extrae la probabilidad de la clase positiva para cada una de las 5 salidas."""
    probs_per_label = clf.predict_proba(X)
    return np.stack([p[:, 1] for p in probs_per_label], axis=1)

def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

test_probs = get_prob_matrix(model, X_test)
test_pred  = (test_probs >= 0.5).astype(int)

metrics_df = compute_metrics(Y_test, test_pred, test_probs)
metrics_df

,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.814643,0.801070,0.777778,0.789252,0.301059
1,Infarto,0.790083,0.715556,0.292727,0.415484,0.295847
2,Isquemia/ST-T,0.825765,0.795918,0.374280,0.509138,0.273654
3,Hipertrofia,0.878128,0.333333,0.003817,0.007547,0.192561
4,Conducción (bloqueos),0.860982,0.837931,0.489919,0.618321,0.226975
5,Macro,0.833920,0.696762,0.387704,0.467948,0.258019


## 10. Registro en la tabla comparativa común



Añade (o actualiza, si ya existía) la fila de este notebook en `model_comparison.csv`, compartido por todos los notebooks de la serie — CNN y modelos clásicos por igual.

In [10]:
macro_row = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0]
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION,
    "Parametros_entrenables": sum(t.tree_.node_count for t in model.estimators_),
    **macro_row.to_dict(),
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV, sep=None, engine="python")
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row

comparison.to_csv(COMPARISON_CSV, index=False)
comparison

,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,01_cnn_simple_sin_filtrado,sin_filtrado,3653,0.823077,0.760360,0.346851,0.413249,0.254433
1,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
2,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
3,04_cnn_profunda_balanceo,con_filtrado,105317,0.810936,0.577661,0.792424,0.656731,0.286431
4,05_cnn_profunda_balanceo_umbral,con_filtrado,105317,0.840871,0.637769,0.722755,0.672034,0.322418
5,07_xgboost,features_clasicas,125290,0.837905,0.708955,0.501942,0.566368,0.200660
6,06_random_forest,features_clasicas,1323262,0.833920,0.696762,0.387704,0.467948,0.258019
